# Langchain LLM

# 選擇模型

## Gemini

In [ ]:
# from langchain_google_genai import ChatGoogleGenerativeAI

# API_KEY = "這邊請改成你自己的API_KEY值"

# model_name = 'gemini-2.5-flash'

# llm = ChatGoogleGenerativeAI(
#     model=model_name,
#     google_api_key=API_KEY
# )

## LM Studio

In [24]:
from langchain_openai import ChatOpenAI
model_name = 'google/gemma-3-4b'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變
# base_url = 'http://localhost:1234/v1'  # 設定 LM Studio 本地伺服器的URL
base_url = 'http://192.168.31.200:1234/v1'

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url 
)

## Ollama

In [ ]:
%%writefile go.sh
#!/bin/bash
set -e  # 若有錯誤立即停止執行

echo "🚀 開始安裝 Ollama..."

# 安裝 Ollama
curl -fsSL https://ollama.com/install.sh | sh

# 重新載入 PATH（確保第一次執行也能找到 ollama）
export PATH=$PATH:/usr/local/bin

# 啟動 Ollama 服務（背景執行）
echo "🟢 啟動 Ollama 服務中..."
ollama serve &

# 等待服務啟動（避免太快執行 pull）
sleep 5

# 拉取指定模型
echo "📦 下載模型 gemma3:12b..."
ollama pull gemma3:12b

# 安裝 Python 套件
echo "🐍 安裝 Python 模組 ollama..."
pip install -q ollama

echo ""
echo "✅ 安裝完畢~~~~~~~"


執行上面的程式後，打開終端機後輸入  
```
sh go.sh
```
需要等終端機內的程式安裝完畢後，才能繼續下面的程式

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gemma-3-12b'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變
base_url = 'http://localhost:11434/v1'  # 設定 LM Studio 本地伺服器的URL

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url 
)

## 測試模型

In [25]:
from langchain_core.messages import HumanMessage
messages = [
    HumanMessage("機器學習的定義")
]
result = llm.invoke(messages)
print(result.content)

機器學習 (Machine Learning, ML) 是一個讓電腦無需明確編程就能學習和改進的領域。簡單來說，它就是**讓電腦從數據中學習模式並做出預測或決策**。

以下是更詳細的解釋：

**核心概念:**

* **數據驅動:** 機器學習的核心是利用大量數據來訓練模型。
* **模式識別:** 模型會分析這些數據，尋找其中的模式、關係和趨勢。
* **自動學習:**  與傳統編程不同，機器學習模型不需要人工明確地告訴它如何解決問題。相反，它通過觀察數據並調整自身參數來“學習”。
* **預測與決策:** 訓練好的模型可以根據新的數據進行預測或做出決策。

**主要類型:**

機器學習通常分為以下幾種類型：

* **監督學習 (Supervised Learning):**  使用標記過的數據（包含輸入和正確的輸出）來訓練模型。
    * **分類 (Classification):** 預測一個輸入屬於哪個類別，例如垃圾郵件檢測、圖像識別。
    * **回歸 (Regression):** 預測一個連續數值，例如股票價格預測、房屋價格估價。
* **非監督學習 (Unsupervised Learning):**  使用未標記的數據來訓練模型，讓模型自己發現數據中的結構和模式。
    * **聚類 (Clustering):** 將數據分成不同的群組，例如客戶分群、異常檢測。
    * **降維 (Dimensionality Reduction):** 減少數據的維度，同時保留重要的信息，例如圖像壓縮。
* **強化學習 (Reinforcement Learning):**  讓一個代理人（Agent）通過與環境互動來學習最佳策略，例如遊戲 AI、機器人控制。

**常見算法:**

機器學習使用各種不同的算法，包括：

* **線性回歸 (Linear Regression)**
* **邏輯迴歸 (Logistic Regression)**
* **決策樹 (Decision Trees)**
* **支持向量機 (Support Vector Machines, SVMs)**
* **K-means 聚類 (K-means Clustering)**
* **神經網絡 (Neural Networks) - 包括深度學習 (De

# Document Embedding
選一個Text Embedding Model

In [ ]:
# # Huggingface Embeddings
# from langchain_huggingface import HuggingFaceEmbeddings

# model_name = "BAAI/bge-large-zh-v1.5"
# embeddings = HuggingFaceEmbeddings(model_name=model_name)

# vector = embeddings.embed_query("What's our Q1 revenue?") # 測試連接
# len(vector)

In [2]:
from langchain.embeddings.base import Embeddings
from openai import OpenAI

class LmStudioEmbeddings(Embeddings):
    def __init__(self, model_name, url):
        self.model_name = model_name
        self.url = url
        self.client = OpenAI(base_url=url, api_key="lm-studio")

    def embed_query(self, text: str):
        response = self.client.embeddings.create(input=text,model=self.model_name)
        return response.data[0].embedding

    def embed_documents(self, texts: list[str]):
        # 回傳多個文件的 embedding
        response = self.client.embeddings.create(input=texts,model=self.model_name)
        return [x.embedding for x in response.data]
        # return [self.model.encode(t).tolist() for t in texts]

# embedding = LmStudioEmbeddings(model_name="text-embedding-bge-large-zh-v1.5", url="http://127.0.0.1:1234/v1")
embeddings = LmStudioEmbeddings(model_name="text-embedding-bge-large-zh-v1.5", url="http://192.168.31.200:1234/v1")
# embeddings.embed_query("What's our Q1 revenue?") # 測試連接

# 載入資料

In [7]:
# 從 langchain_community 套件中匯入需要的 Loader 類別
from langchain_community.document_loaders import DirectoryLoader  # 用來載入整個資料夾
from langchain_community.document_loaders import TextLoader       # 用來載入單一 txt 檔案

# 建立 DirectoryLoader 的實例，用來載入整個資料夾中的文件
loader = DirectoryLoader(
    path="cv_data",      # 指定資料夾名稱，這裡是 "qa_data"
    glob="**/*.txt",     # 使用 glob 模式，支援遞迴搜尋所有子資料夾的 txt 檔案
    loader_cls=TextLoader,           # 指定每個檔案使用 TextLoader 來讀取
    loader_kwargs={"encoding": "utf-8"}  # 傳給 TextLoader 的參數，指定檔案編碼為 UTF-8
)

# 使用 load() 方法載入資料夾中的所有文件
# 這會回傳一個 List，裡面每個元素都是一個 Document 物件
documents = loader.load()

# 列印總共載入了多少筆資料
print(f"總共有{len(documents)}筆資料")

總共有4筆資料


# 文本分段

In [8]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

split_docs = text_splitter.split_documents(documents)

print(f"原始文件數量：{len(documents)}")
print(f"分段後文件數量：{len(split_docs)}")

for chunk in split_docs:
    print('-------')
    print(chunk.page_content)
    print(chunk.metadata)


原始文件數量：4
分段後文件數量：15
-------
姓名：陳仁政
出生年份：1972

學術與專業背景：
博士學位｜中原大學電子研究所
博士後研究｜中央研究院資訊科學研究所


產業與教育經歷：
資深工程師｜吉鴻電子
正工程師｜冠捷科技
資料科學家｜104人力銀行人資學院
兼任實務教師｜長庚大學工商管理系
顧問｜104人力銀行人資學院


現職：
講師｜台灣人工智能產業協會
講師｜實踐大學推廣中心
講師｜緯育TibaMe
{'source': 'cv_data\\CV_01.txt'}
-------
現職：
講師｜台灣人工智能產業協會
講師｜實踐大學推廣中心
講師｜緯育TibaMe


專案經驗：
智慧金融｜收鈔機韌體開發（偽鈔偵測）
消費電子｜電視韌體開發
  智慧交通與監控
  高承載管制違規偵測系統
  手持式雷射測距儀
  區間測速系統
  違停偵測系統
  活動地磅系統
  測速照相系統
AI與人力資源分析｜人才適任與久任度評估系統
{'source': 'cv_data\\CV_01.txt'}
-------
專長領域：
人工智慧與機器學習｜機器學習、生成式 AI、大語言模型（LLM）
電腦視覺與影像處理｜影像識別、智慧監控系統開發
嵌入式系統與數位電路｜數位電路設計、嵌入式系統開發
軟體開發與工程｜程式設計、多平台軟硬體整合
{'source': 'cv_data\\CV_01.txt'}
-------
課程資訊：
實踐大學推廣部 人工智慧與大數據應用全修班 2020/08
華岡興業基金會 AI人力資源大數據分析課程 2021/07
實踐大學推廣部 Python與資料處理實戰訓練班 2021/11
金融研訓院 AI與大數據在HR專業實務的應用workshop 2022/04
AutoML AI人工智慧科技數據分析人才培育暨認證師資培訓教師研習會 2022/07 (東海大學、元智大學、世新大學、真理大學、長庚大學)
長庚大學工商系碩士班 商業資料科學實務與應用 2022
中華郵政AI Workshop 2022/11
長庚醫院 AutoML Workshop 2023-04
南亞科技 ChatGPT技術原理及應用 2023-07
長庚醫院 AI計畫撰寫工作坊 2023-07
人工智慧產業協會 AI人力資源大數據分析課程 2023/07
實踐大學

# 將資料存進向量資料庫(Chroma)
* https://python.langchain.com/v0.1/docs/modules/data_connection/vectorstores/
* https://python.langchain.com/v0.1/docs/integrations/vectorstores/chroma/

In [9]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
import uuid

# 建立或載入現有的 Chroma 向量資料庫
vector_store = Chroma(
    collection_name=str(uuid.uuid4()),     # collection 名稱（相當於一個資料表）
    embedding_function=embeddings,        # 指定嵌入函式
    # persist_directory=persist_dir         # 向量資料儲存路徑
)


In [10]:
# 新增文件資料
vector_store.add_documents(split_docs)
print("✅ 成功新增新資料至 Chroma。")

✅ 成功新增新資料至 Chroma。


In [11]:
# 尋找相關的文件
vector_store.similarity_search("陳仁政的教學經驗", k=5)

[Document(id='4517aa84-4022-4178-81b2-1e89d709627e', metadata={'source': 'cv_data\\CV_01.txt'}, page_content='姓名：陳仁政\n出生年份：1972\n\n學術與專業背景：\n博士學位｜中原大學電子研究所\n博士後研究｜中央研究院資訊科學研究所\n\n\n產業與教育經歷：\n資深工程師｜吉鴻電子\n正工程師｜冠捷科技\n資料科學家｜104人力銀行人資學院\n兼任實務教師｜長庚大學工商管理系\n顧問｜104人力銀行人資學院\n\n\n現職：\n講師｜台灣人工智能產業協會\n講師｜實踐大學推廣中心\n講師｜緯育TibaMe'),
 Document(id='f1f85b0b-c0d9-460f-82d5-8d92f7429476', metadata={'source': 'cv_data\\CV_01.txt'}, page_content='現職：\n講師｜台灣人工智能產業協會\n講師｜實踐大學推廣中心\n講師｜緯育TibaMe\n\n\n專案經驗：\n智慧金融｜收鈔機韌體開發（偽鈔偵測）\n消費電子｜電視韌體開發\n  智慧交通與監控\n  高承載管制違規偵測系統\n  手持式雷射測距儀\n  區間測速系統\n  違停偵測系統\n  活動地磅系統\n  測速照相系統\nAI與人力資源分析｜人才適任與久任度評估系統'),
 Document(id='2d73dd87-f848-42f4-b369-adc952c8f8aa', metadata={'source': 'cv_data\\CV_03.txt'}, page_content='現職：\n顧問｜智慧醫療新創公司\n講師｜長庚醫院教育訓練中心\n講師｜台灣醫療資訊學會\n\n專案經驗：\n智慧醫療\n- 臨床決策支援系統（CDSS）\n- 醫學影像輔助診斷系統\n- 病歷資料結構化與分析\n\n健康科技\n- 慢性病風險預測模型\n- 醫療 AI 模型驗證與法規對應\n- 醫療資料匿名化與隱私保護系統'),
 Document(id='31e72a6f-4244-494f-b0a1-60c0705302e0', metadata={'source': 'cv_data\\CV_04.txt'}, page

# 建立RAG Chain

In [27]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = '''
你是一位履歷檔案管理人，會根據以下履歷檔案，回答使用者的問題：
{summaries}
'''

prompt_template = ChatPromptTemplate(
    messages= [
            ("system", system_prompt),
            ("user", "{question}")
        ]
    )

def get_cv(question):
    cv_docs = vector_store.similarity_search(question, k=5)
    contents = [doc.page_content for doc in cv_docs]
    # log
    print("找到的文件內容為：") #log
    for doc in contents:
        print(doc)
        print('--')
    print("-"*50) #log
    return {"question":question, "summaries":contents}

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
rag_chain = RunnableLambda(get_cv) | prompt_template | llm | StrOutputParser()

In [28]:
for chunk in rag_chain.stream("請問陳仁政的專長是什麼？"):
    print(chunk, end="")

找到的文件內容為：
姓名：陳仁政
出生年份：1972

學術與專業背景：
博士學位｜中原大學電子研究所
博士後研究｜中央研究院資訊科學研究所


產業與教育經歷：
資深工程師｜吉鴻電子
正工程師｜冠捷科技
資料科學家｜104人力銀行人資學院
兼任實務教師｜長庚大學工商管理系
顧問｜104人力銀行人資學院


現職：
講師｜台灣人工智能產業協會
講師｜實踐大學推廣中心
講師｜緯育TibaMe
--
現職：
顧問｜智慧醫療新創公司
講師｜長庚醫院教育訓練中心
講師｜台灣醫療資訊學會

專案經驗：
智慧醫療
- 臨床決策支援系統（CDSS）
- 醫學影像輔助診斷系統
- 病歷資料結構化與分析

健康科技
- 慢性病風險預測模型
- 醫療 AI 模型驗證與法規對應
- 醫療資料匿名化與隱私保護系統
--
現職：
技術顧問｜工研院智慧製造中心
講師｜金屬工業研究發展中心
講師｜中華民國自動化科技學會

專案經驗：
智慧製造
- 半導體製程設備自動化整合
- 生產線 OEE 與良率即時監控系統
- 機械手臂視覺定位與抓取系統

工業物聯網（IIoT）
- 工廠設備感測資料蒐集與分析
- Edge AI 製程異常預警系統
--
專長領域：
醫療資訊｜HIS、EMR、FHIR
醫療 AI｜影像診斷、預測模型
健康資料分析｜臨床資料、穿戴裝置
醫療法規與資料治理｜AI 醫療合規

課程資訊：
醫療資訊系統實務 2018
醫療大數據分析 2019
AI 輔助醫學影像診斷 2021
智慧醫療專案規劃與導入 2022
生成式 AI 在醫療文件與病歷應用 2024
--
現職：
講師｜台灣人工智能產業協會
講師｜實踐大學推廣中心
講師｜緯育TibaMe


專案經驗：
智慧金融｜收鈔機韌體開發（偽鈔偵測）
消費電子｜電視韌體開發
  智慧交通與監控
  高承載管制違規偵測系統
  手持式雷射測距儀
  區間測速系統
  違停偵測系統
  活動地磅系統
  測速照相系統
AI與人力資源分析｜人才適任與久任度評估系統
--
--------------------------------------------------
陳仁政的專長領域如下：

*   **醫療資訊**：HIS、EMR、FHIR
*   **醫療 AI**：影像診斷、預測模型
*   **健康資料分析**：臨床資料、穿戴裝置

In [29]:
for chunk in rag_chain.stream("請問陳仁政教過的課程有哪些？"):
    print(chunk, end="")

找到的文件內容為：
姓名：陳仁政
出生年份：1972

學術與專業背景：
博士學位｜中原大學電子研究所
博士後研究｜中央研究院資訊科學研究所


產業與教育經歷：
資深工程師｜吉鴻電子
正工程師｜冠捷科技
資料科學家｜104人力銀行人資學院
兼任實務教師｜長庚大學工商管理系
顧問｜104人力銀行人資學院


現職：
講師｜台灣人工智能產業協會
講師｜實踐大學推廣中心
講師｜緯育TibaMe
--
現職：
顧問｜智慧醫療新創公司
講師｜長庚醫院教育訓練中心
講師｜台灣醫療資訊學會

專案經驗：
智慧醫療
- 臨床決策支援系統（CDSS）
- 醫學影像輔助診斷系統
- 病歷資料結構化與分析

健康科技
- 慢性病風險預測模型
- 醫療 AI 模型驗證與法規對應
- 醫療資料匿名化與隱私保護系統
--
現職：
顧問｜金融科技創新園區
講師｜金融研訓院
講師｜實踐大學推廣教育中心

專案經驗：
金融科技
- 個人信用評分模型建置
- 數位銀行風險控管系統
- 即時交易異常偵測平台

風險管理
- 信用風險與市場風險模型
- 反洗錢（AML）與詐欺偵測系統
- 金融法遵自動化系統
--
現職：
講師｜台灣人工智能產業協會
講師｜實踐大學推廣中心
講師｜緯育TibaMe


專案經驗：
智慧金融｜收鈔機韌體開發（偽鈔偵測）
消費電子｜電視韌體開發
  智慧交通與監控
  高承載管制違規偵測系統
  手持式雷射測距儀
  區間測速系統
  違停偵測系統
  活動地磅系統
  測速照相系統
AI與人力資源分析｜人才適任與久任度評估系統
--
專長領域：
醫療資訊｜HIS、EMR、FHIR
醫療 AI｜影像診斷、預測模型
健康資料分析｜臨床資料、穿戴裝置
醫療法規與資料治理｜AI 醫療合規

課程資訊：
醫療資訊系統實務 2018
醫療大數據分析 2019
AI 輔助醫學影像診斷 2021
智慧醫療專案規劃與導入 2022
生成式 AI 在醫療文件與病歷應用 2024
--
--------------------------------------------------
根據提供的履歷檔案，陳仁政教過的課程有：

*   醫療資訊系統實務 (2018)
*   醫療大數據分析 (2019)
*   AI 輔助醫學影像診斷 (2021)
*   智慧醫療專案規劃與導入 (2022)